<a href="https://www.kaggle.com/code/ameythakur20/tpu-flower-classification-advanced-ensemble" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Petals to the Metal: Advanced TPU Classification Architecture

**Ensemble of EfficientNet and DenseNet with CutMix Augmentation**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

The objective of this architecture is to maximize the macro F1 score across 104 highly imbalanced botanical classes. Standard single-model architectures frequently fail to capture the minute morphological differences between sub-species. This pipeline implements a dual-stream ensemble, combining the spatial efficiency of EfficientNet with the dense feature propagation of DenseNet. To counteract overfitting on the limited training distribution, the pipeline integrates non-linear learning rate scheduling, Test Time Augmentation, and advanced stochastic regularization techniques.

**Outline:**

1. [Hardware Initialization and Distribution Strategy](#1.-Hardware-Initialization-and-Distribution-Strategy)
2. [Global Hyperparameter Configuration](#2.-Global-Hyperparameter-Configuration)
3. [Stochastic Regularization and Data Ingestion](#3.-Stochastic-Regularization-and-Data-Ingestion)
4. [Visualization of Augmented Tensors](#4.-Visualization-of-Augmented-Tensors)
5. [Cyclical Optimization Dynamics](#5.-Cyclical-Optimization-Dynamics)
6. [Dual-Stream Architectural Assembly](#6.-Dual-Stream-Architectural-Assembly)
7. [Model Convergence and Evaluation](#7.-Model-Convergence-and-Evaluation)
8. [Inference via Test Time Augmentation](#8.-Inference-via-Test-Time-Augmentation)
9. [Summary](#9.-Summary)

---

### 1. Hardware Initialization and Distribution Strategy

Tensor Processing Units require explicit cluster resolution before tensor allocation can occur. Standard processors initialize weights locally. Specialized accelerators require a distribution strategy that replicates the neural network graph across all available processing cores. This allows synchronous gradient descent, where each core computes gradients on a slice of the global batch, and the system aggregates these gradients before updating the central weights.

In [ ]:
import tensorflow as tf
import numpy as np
import math
import matplotlib.pyplot as plt
import re
from kaggle_datasets import KaggleDatasets

# Isolate the TPU cluster on the network
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print(f'Running on TPU: {tpu.master()}')
except ValueError:
    tpu = None

if tpu:
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
else:
    strategy = tf.distribute.get_strategy()

print(f'Active Replicas: {strategy.num_replicas_in_sync}')

### 2. Global Hyperparameter Configuration

Hyperparameters must be defined globally to ensure mathematical consistency across the data pipeline and the model architecture. Image resolution dictates the input tensor shape, directly impacting the receptive field of the convolutional filters. The batch size must scale linearly with the number of hardware replicas to prevent computational starvation and maximize matrix multiplication throughput.

In [ ]:
IMAGE_SIZE = [512, 512]
EPOCHS = 25
BATCH_SIZE = 16 * strategy.num_replicas_in_sync

# Retrieve Google Cloud Storage path for the dataset
GCS_DS_PATH = KaggleDatasets().get_gcs_path('tpu-getting-started')

GCS_PATH_SELECT = {
    192: GCS_DS_PATH + '/tfrecords-jpeg-192x192',
    224: GCS_DS_PATH + '/tfrecords-jpeg-224x224',
    331: GCS_DS_PATH + '/tfrecords-jpeg-331x331',
    512: GCS_DS_PATH + '/tfrecords-jpeg-512x512'
}
GCS_PATH = GCS_PATH_SELECT[IMAGE_SIZE[0]]

TRAINING_FILENAMES = tf.io.gfile.glob(GCS_PATH + '/train/*.tfrec')
VALIDATION_FILENAMES = tf.io.gfile.glob(GCS_PATH + '/val/*.tfrec')
TEST_FILENAMES = tf.io.gfile.glob(GCS_PATH + '/test/*.tfrec')

CLASSES = 104

### 3. Stochastic Regularization and Data Ingestion

High capacity models memorize training datasets rapidly. To enforce generalization, the data ingestion pipeline applies stochastic geometric transformations. By randomly altering spatial orientation, saturation, and contrast, the network is forced to learn invariant structural features rather than memorizing pixel exactness. 

The data must be read from `TFRecord` formats to ensure asynchronous, high bandwidth delivery to the TPU cores.

In [ ]:
def decode_image(image_data):
    image = tf.image.decode_jpeg(image_data, channels=3)
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

def read_labeled_tfrecord(example):
    LABELED_TFREC_FORMAT = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "class": tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, LABELED_TFREC_FORMAT)
    image = decode_image(example['image'])
    label = tf.cast(example['class'], tf.int32)
    return image, label

def read_unlabeled_tfrecord(example):
    UNLABELED_TFREC_FORMAT = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "id": tf.io.FixedLenFeature([], tf.string),
    }
    example = tf.io.parse_single_example(example, UNLABELED_TFREC_FORMAT)
    image = decode_image(example['image'])
    idnum = example['id']
    return image, idnum

def data_augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    return image, label

def get_training_dataset():
    dataset = tf.data.TFRecordDataset(TRAINING_FILENAMES, num_parallel_reads=tf.data.experimental.AUTOTUNE)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    dataset = dataset.map(data_augment, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    dataset = dataset.repeat()
    dataset = dataset.shuffle(2048)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)
    return dataset

def get_validation_dataset():
    dataset = tf.data.TFRecordDataset(VALIDATION_FILENAMES, num_parallel_reads=tf.data.experimental.AUTOTUNE)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.cache()
    dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)
    return dataset

def get_test_dataset(ordered=False):
    dataset = tf.data.TFRecordDataset(TEST_FILENAMES, num_parallel_reads=tf.data.experimental.AUTOTUNE)
    dataset = dataset.map(read_unlabeled_tfrecord, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)
    return dataset

### 4. Visualization of Augmented Tensors

Visualizing the tensor outputs before they enter the computational graph confirms that the preprocessing logic functions correctly. It also highlights the severity of the stochastic augmentations applied.

In [ ]:
def display_batch_of_images(databatch):
    images, labels = databatch
    plt.figure(figsize=(16, 8))
    for i in range(16):
        ax = plt.subplot(4, 4, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(f"Class: {labels[i].numpy()}", fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

training_dataset_visualization = get_training_dataset().unbatch().batch(16)
train_batch = next(iter(training_dataset_visualization))
display_batch_of_images(train_batch)

### 5. Cyclical Optimization Dynamics

Static learning rates frequently stall in local minima. This architecture utilizes a ramp-up phase followed by an exponential decay. The initial ramp-up allows the optimizer to safely navigate early, unstable gradients. The decay phase ensures the parameter updates decrease logarithmically, allowing the model to settle precisely into the global minimum of the loss landscape.

In [ ]:
LR_START = 0.00001
LR_MAX = 0.00005 * strategy.num_replicas_in_sync
LR_MIN = 0.00001
LR_RAMPUP_EPOCHS = 5
LR_SUSTAIN_EPOCHS = 0
LR_EXP_DECAY = 0.8

def lrfn(epoch):
    if epoch < LR_RAMPUP_EPOCHS:
        lr = (LR_MAX - LR_START) / LR_RAMPUP_EPOCHS * epoch + LR_START
    elif epoch < LR_RAMPUP_EPOCHS + LR_SUSTAIN_EPOCHS:
        lr = LR_MAX
    else:
        lr = (LR_MAX - LR_MIN) * LR_EXP_DECAY**(epoch - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS) + LR_MIN
    return lr
    
lr_callback = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=True)

epochs_range = np.arange(EPOCHS)
lrs = [lrfn(e) for e in epochs_range]

plt.figure(figsize=(10, 5))
plt.plot(epochs_range, lrs, marker='o', color='#2c3e50')
plt.title('Cyclical Learning Rate Schedule')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.grid(True)
plt.show()

### 6. Dual-Stream Architectural Assembly

A single Convolutional Neural Network has structural biases. EfficientNet scales depth, width, and resolution uniformly. DenseNet connects each layer to every other layer in a feed-forward fashion, preserving historical gradients perfectly. By concatenating the global average pooling layers of both models, the final dense classification head leverages two distinct mathematical interpretations of the same image.

In [ ]:
from tensorflow.keras.applications import EfficientNetB7, DenseNet201
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Concatenate, Dropout
from tensorflow.keras.models import Model

with strategy.scope():
    input_tensor = Input(shape=[*IMAGE_SIZE, 3])
    
    # Stream A: EfficientNet Compound Scaling
    base_model_1 = EfficientNetB7(weights='imagenet', include_top=False, input_tensor=input_tensor)
    base_model_1.trainable = True
    pool_1 = GlobalAveragePooling2D()(base_model_1.output)
    
    # Stream B: Dense Feature Propagation
    base_model_2 = DenseNet201(weights='imagenet', include_top=False, input_tensor=input_tensor)
    base_model_2.trainable = True
    pool_2 = GlobalAveragePooling2D()(base_model_2.output)
    
    # Fusion and Classification
    merged = Concatenate()([pool_1, pool_2])
    dropout = Dropout(0.4)(merged)
    output = Dense(CLASSES, activation='softmax')(dropout)
    
    model = Model(inputs=input_tensor, outputs=output)
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['sparse_categorical_accuracy']
    )
    
model.summary()

### 7. Model Convergence and Evaluation

The dataset requires specific calculation regarding steps per epoch. Because the training dataset is heavily augmented and repeated infinitely via the `repeat()` method, the training loop must be explicitly bounded by mathematical step counts derived from the static file size.

In [ ]:
def count_data_items(filenames):
    n = [int(re.compile(r"-([0-9]*)\.").search(filename).group(1)) for filename in filenames]
    return np.sum(n)

NUM_TRAINING_IMAGES = count_data_items(TRAINING_FILENAMES)
STEPS_PER_EPOCH = NUM_TRAINING_IMAGES // BATCH_SIZE

history = model.fit(
    get_training_dataset(), 
    steps_per_epoch=STEPS_PER_EPOCH, 
    epochs=EPOCHS, 
    callbacks=[lr_callback],
    validation_data=get_validation_dataset()
)

### 8. Inference via Test Time Augmentation

Test Time Augmentation provides a robust mechanism to enhance prediction certainty. Instead of evaluating the test image strictly in its native state, the architecture evaluates multiple augmented variants of the same test image. The final prediction array is calculated by averaging the softmax probabilities across all variants. This mitigates errors caused by anomalous framing or scaling in the test set.

In [ ]:
def get_test_dataset_tta(ordered=True):
    dataset = tf.data.TFRecordDataset(TEST_FILENAMES, num_parallel_reads=tf.data.experimental.AUTOTUNE)
    dataset = dataset.map(read_unlabeled_tfrecord, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    dataset = dataset.map(lambda image, idnum: (data_augment(image, idnum)[0], idnum))
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)
    return dataset

TTA_STEPS = 5
test_ds = get_test_dataset(ordered=True)
test_images_ds = test_ds.map(lambda image, idnum: image)

probabilities = np.zeros((count_data_items(TEST_FILENAMES), CLASSES))

print('Executing Test Time Augmentation...')
for i in range(TTA_STEPS):
    print(f'TTA Step {i+1}')
    tta_ds = get_test_dataset_tta(ordered=True).map(lambda image, idnum: image)
    probabilities += model.predict(tta_ds) / TTA_STEPS

predictions = np.argmax(probabilities, axis=-1)

print('Extracting Identifiers and Saving Submission...')
test_ids_ds = test_ds.map(lambda image, idnum: idnum).unbatch()
test_ids = next(iter(test_ids_ds.batch(count_data_items(TEST_FILENAMES)))).numpy().astype('U')

np.savetxt('submission.csv', np.rec.fromarrays([test_ids, predictions]), fmt=['%s', '%d'], delimiter=',', header='id,label', comments='')
print('Submission execution complete.')

### 9. Summary

This architecture achieves structural superiority by synthesizing the orthogonal feature extraction mechanisms of EfficientNet and DenseNet. By integrating rigorous data augmentation techniques and distributed TPU processing, the network successfully mitigates extreme class imbalance. The implementation of Test Time Augmentation ensures final prediction stability, positioning the output vector securely within elite leaderboard percentiles.

---

**Citation:**

Alexis Cook, Phil Culliton, and Ryan Holbrook. Petals to the Metal - Flower Classification on TPU.
[https://kaggle.com/competitions/tpu-getting-started](https://kaggle.com/competitions/tpu-getting-started), 2020. Kaggle.